In [102]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from category_encoders.target_encoder import TargetEncoder
from xgboost import XGBClassifier
from skopt.space import Real, Categorical, Integer
from skopt import BayesSearchCV
#pip install xgboost
#pip install --upgrade category_encoders
#pip install scikit-optimize

df = pd.read_csv('bank-additional-full.csv', delimiter=';')
#getting rid of features that are not very useful
cols_to_drop = ['duration', 'emp.var.rate', 'cons.price.idx', 'cons.conf.idx', 'euribor3m', 'nr.employed', 'poutcome', 'default']
#rename columns so they are undersetandable
df.head()


df = df.drop(columns=cols_to_drop).rename(columns={'job': 'job_type', 
                                                   'housing': 'housing_loan_status', 'loan': 'personal_loan_status', 
                                                   'contact': 'contact_type', 'month': 'contact_month', 
                                                   'day_of_week': 'contact_day_of_week', 'campaign': 'num_contacts', 
                                                   'pdays': 'days_last_contact', 'previous': 'previous_contacts', 
                                                   'y': 'result'
                                                    })
# convert the target to numerical values
df['result'] = df['result'].replace({'yes': 1, 'no': 0})
df['result'] = pd.to_numeric(df['result'], errors='coerce')

# 2. Drop any rows where 'result' is now NaN (missing)
df = df.dropna(subset=['result'])

# 3. Force the column to be a pure Integer (not float)
df['result'] = df['result'].astype(int)
df['result'].value_counts()


#Can use XGBRegressor instead for regression models
cat_cols = ['job_type', 'marital', 'education', 'housing_loan_status', 'personal_loan_status', 'contact_type', 'contact_month', 'contact_day_of_week']
#te = TargetEncoder(cat_cols)
#df[cat_cols] = te.fit_transform(df[cat_cols], df['result'])

df['result'].value_counts()



X = df.drop(columns='result')
y = df['result']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=8)


estimators = [
    ('encoder', TargetEncoder(cols=cat_cols)),
    ('clf', XGBClassifier(random_state=8)) # can customize objective function with the objective parameter
]
pipe = Pipeline(steps=estimators)
pipe
te = pipe.named_steps['encoder']
encoded_sample = te.fit_transform(X_train, y_train)

search_space = {
    'clf__max_depth': Integer(2,8),
    'clf__learning_rate': Real(0.001, 1.0, prior='log-uniform'),
    'clf__subsample': Real(0.5, 1.0),
    'clf__colsample_bytree': Real(0.5, 1.0),
    'clf__colsample_bylevel': Real(0.5, 1.0),
    'clf__colsample_bynode' : Real(0.5, 1.0),
    'clf__reg_alpha': Real(0.0, 10.0),
    'clf__reg_lambda': Real(0.0, 10.0),
    'clf__gamma': Real(0.0, 10.0)
}

opt = BayesSearchCV(pipe, search_space, cv=3, n_iter=10, scoring='roc_auc', random_state=8) 
opt.fit(X_train, y_train)

from sklearn.utils.multiclass import type_of_target

print(f"Dtype of y_train: {y_train.dtype}")
print(f"Unique values in y_train: {y_train.unique()}")
print(f"Format detected by sklearn: {type_of_target(y_train)}")


Dtype of y_train: int64
Unique values in y_train: [0 1]
Format detected by sklearn: binary
